# NB-10 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-10**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-10.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This continues NB-09 with a hidden topology. It distinguishes beliefs available sequentially from retrospective smoothing after later observations arrive.

- [Volume VI, Chapter 3 — Sequential inference under a knowledge-time filtration](https://sherandoresearch.github.io/power-markets-series/vol6_inference_and_decisions.html#c3)
- [Volume IV, Temporal interlude T5 — Hidden operative topology](https://sherandoresearch.github.io/power-markets-series/vol4_testbed_atlas.html#temp-t5)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-10', require_key=True)
print(setup_status)


# NB-10 — Hidden topology and sequential beliefs
Companion to Volume VI §§1–3 and the Volume IV T5 temporal experiment.
Prerequisite: NB-09. The candidate topology is now hidden from the estimator.
We read the existing 120-hour observations and candidate conditional means;
no noise, regime schedule or synthetic data is generated.

The question is what can be inferred at each observation time. A switching
filter uses a prefix; a smoother and Viterbi path use the whole retained panel.
The conditional means, transition probabilities and noise scales are declared
toy-model inputs, not estimates fitted to a real ISO's data.

In [ ]:
NOTEBOOK_ID='NB-10'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return response['result']
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

print('Local retained support:',inputs['raw_inspection'])
r=dict(inputs['original_model'],**inputs['evaluation_only']);hours=np.asarray(inputs['hours'])
values={}
def run_saved(name,parents=(),request=None):
    entry=inputs['requests'][name]
    values[name]=execute(name,entry['method'],request if request is not None else entry['request'],parents)
    return values[name]
result={}
for key in ('direct','filter','viterbi'):
    value=run_saved(key+':complete')['regime_inference_result']
    if key=='viterbi':result[key]=np.asarray(value['viterbi_state_indices'])
    else:result[key]=np.asarray(value['probabilities'])
    if key=='filter':
        result['predicted']=np.asarray(value['predicted_probabilities']);result['ll']=np.asarray(value['log_likelihood'])
for key,field in (('smooth','retrospective_regime_marginals'),('fixed','regime_inference_result')):
    value=run_saved(key+':prefix:120')[field]
    result[key]=np.asarray(value['probabilities'])
print('Retained observations:',len(hours),'channels:',r['feature_labels'])
print('Declared noise standard deviations:',r['observation_std'])
print('Likelihoods, filter probabilities and Viterbi states returned by private execution.')

## Observation likelihood, switching filter and retrospective inference
The spread channel can fail to distinguish the candidates when both predict
zero congestion, while flow sensors can still separate them. Conditional on
observed demand, likelihoods compare both candidate means; the true state never
enters the inference function. The direct comparator uses equal per-hour priors.
The fixed-regime comparator updates one unknown but unchanging regime from the
same observations. It cannot represent an outage followed by restoration.

The forward-backward smoother below is a new deterministic calculation on the
retained observations. It reports marginal probabilities, unlike Viterbi's
most probable joint state path. Both are **ex post**, not usable predictors at
the original hourly cutoff.

In [ ]:
fig,axes=plt.subplots(2,1,figsize=(12,7),sharex=True)
for key,label,color in [('direct','Direct hourly likelihood','#B48EAD'),('fixed','Fixed-regime Bayes','#EBCB8B'),('filter','Switching filter','#88C0D0')]:
    axes[0].plot(hours,result[key][:,1],label=label,color=color)
axes[0].set(ylabel='Outage probability',title='Prefix-only estimates under different model assumptions');axes[0].legend(ncol=3,fontsize=8)
axes[1].plot(hours,result['filter'][:,1],label='Filter',color='#88C0D0')
axes[1].plot(hours,result['smooth'][:,1],label='Smoother · ex post',color='#A3BE8C')
axes[1].step(hours,result['viterbi'],where='post',label='Viterbi · ex post',color='#D08770',alpha=.7)
axes[1].set(xlabel='Retained hour',ylabel='Probability / state',title='Retrospective information changes the answer');axes[1].legend(ncol=3,fontsize=8)
fig.tight_layout();plt.show()
ll=result['ll']
for end in range(1,len(hours)+1):
    prefix=result['filter'] if end==120 else np.asarray(run_saved(f'filter:prefix:{end}')['regime_inference_result']['probabilities'])
    np.testing.assert_allclose(prefix,result['filter'][:end],atol=1e-13)
np.testing.assert_allclose(result['smooth'][-1],result['filter'][-1],atol=1e-12)
print('All 120 filter prefixes agree with the corresponding full-run prefix.')
print('Largest filter/smoother difference:',round(float(np.max(abs(result['filter']-result['smooth']))),4))

## Evaluate only after inference
Hidden truth is used here for scoring, not estimation. Five fixed probability
bins display their actual sample counts. These are descriptive diagnostics of
one dependent 120-hour trajectory, not independent calibration trials or a
demonstration of general predictive skill. No fitted calibration or confidence
interval is claimed. The baseline that always predicts intact is shown too.

In [ ]:
truth=np.asarray(r['true_state']);metrics={};scored={}
for key in ('fixed','direct','filter','smooth','always_intact'):
    name='score:'+key;q=copy.deepcopy(inputs['requests'][name]['request'])
    parents=[]
    if key!='always_intact':
        q['inputs']['predictions_and_truth']['probabilities']=result[key][:,1].tolist()
        parents=[key+':prefix:120' if key in ('fixed','smooth') else key+':complete']
    output=run_saved(name,parents,q)['forecast_evaluation'];scored[key]=output
    if key!='always_intact':
        metrics[key]=output['metrics'];print(key,{k:metrics[key][k] for k in ('accuracy','brier_score','false_positive_count','false_negative_count')})
print('Always intact:',scored['always_intact']['metrics']['brier_score'],'Brier; outage hours:',int(truth.sum()))
print('Source scoring clips before Brier and log loss; reliability bins use unclipped probabilities.')
bins=scored['filter']['reliability_bins']
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot([0,1],[0,1],'--',color='#4C566A')
axes[0].plot([b['mean_p'] for b in bins],[b['frequency'] for b in bins],'o-',color='#88C0D0')
for b in bins:axes[0].annotate('n='+str(b['n']),(b['mean_p'],b['frequency']),xytext=(0,9),textcoords='offset points',ha='center',fontsize=8)
axes[0].set(xlabel='Mean filtered probability',ylabel='Observed outage frequency',ylim=(-.08,1.15),title='Descriptive reliability · five fixed bins')
labels=list(metrics);axes[1].bar(labels,[metrics[k]['brier_score'] for k in labels],color=['#EBCB8B','#B48EAD','#88C0D0','#A3BE8C'])
axes[1].set(ylabel='Brier score',title='Same retained trajectory · smoother is ex post');fig.tight_layout();plt.show()

## Existing delayed public status: event time differs from knowledge time
The retained case supplies a public-status channel that changes four hours
after each physical outage begins, and two or three hours after restoration.
The table below pairs the existing physical and published change hours. At
hour 33 the public channel still reports intact even though physical truth has
changed. That truth is visible only in this evaluation view.

This fixture records publication hours, not an independently observed collector
receipt clock. We cannot equate publication with actual workbench knowledge.
The public-status channel is plotted as a separate comparator; it is **not**
assimilated as a perfectly reliable measurement or backdated to physical time.
A delayed-measurement filter with receipt vintages remains a separate extension.

In [ ]:
for lag in r['public_state_lags']:print(lag)
public=np.asarray(r['public_state']);assert truth[33]==1 and public[33]==0
fig,ax=plt.subplots(figsize=(12,4))
ax.step(hours,truth,where='post',label='Physical truth · evaluation only',color='#D08770')
ax.step(hours,public,where='post',label='Retained published status',color='#EBCB8B')
ax.plot(hours,result['filter'][:,1],label='Filter from flow/price observations',color='#88C0D0')
ax.set(xlabel='Retained hour',ylabel='State / outage probability',title='Publication lag is not a physical-state lag');ax.legend(fontsize=8);fig.tight_layout();plt.show()

## Excellent fit can still mean incorrect physical attribution
Recompute the existing dictionary-mismatch example: an outage-state L13 shadow
price generates the congestion field. The stale intact dictionary explains the
same prices using the wrong constraints. A small reconstruction residual alone
does not identify topology, directional constraints or physical causes.
The spread-only candidate means also coincide during some retained hours.

In [ ]:
d=copy.deepcopy(inputs['dictionary_explanatory_truth'])
for label,which in (('matched_outage','matched'),('stale_intact','stale')):
    value=run_saved('dictionary:'+label)['nonnegative_dictionary_fit']
    d['matched_dictionary_coefficients' if which=='matched' else 'stale_intact_dictionary_coefficients']=value['coefficients']
    d[which+'_fit']=value['fitted_values'];d[which+'_residual_norm']=value['residual_norm'];d[which+'_support']=value['support_ids']
means=np.asarray(r['state_conditional_means'])
ambiguous=np.flatnonzero(np.isclose(means[:,0,1],means[:,1,1],atol=1e-9))
print('Spread-indistinguishable hours:',len(ambiguous),'of',len(hours))
print('Matched support:',d['matched_support'],'stale support:',d['stale_support'])
print('Matched residual:',d['matched_residual_norm'],'stale residual:',d['stale_residual_norm'])
fig,axes=plt.subplots(1,2,figsize=(12,4));x=np.arange(3)
for offset,key,label,color in [(-.25,'true_shadow_prices','True outage shadow','#A3BE8C'),(0,'matched_dictionary_coefficients','Matched fit','#88C0D0'),(.25,'stale_intact_dictionary_coefficients','Stale fit','#D08770')]:
    axes[0].bar(x+offset,d[key],.25,label=label,color=color)
axes[0].set_xticks(x,d['line_labels']);axes[0].set(ylabel='Toy currency/MWh',title='Different inferred constraint coefficients');axes[0].legend(fontsize=8)
for key,label,color in [('true_congestion_price_field','True field','#A3BE8C'),('stale_fit','Stale dictionary fit','#D08770')]:
    axes[1].plot(x,d[key],'o--',label=label,color=color)
axes[1].set_xticks(x,['B1','B2','B3']);axes[1].set(ylabel='Toy currency/MWh',title='Nearly identical nodal price field');axes[1].legend();fig.tight_layout();plt.show()

## Exercise and boundary
At a restoration hour, compare the filter, published status and smoother.
Which observations could each have used? Explain why the better retrospective
state score does not establish a better implementable trading rule.
This notebook demonstrates inference under a declared model. It does not infer
real PJM topology from LMP jumps, fit an ISO transition model, or use withheld
state labels as input features.

In [ ]:
for control in inputs['negative_controls']:reject(control['method'],control['request'])
checked=dict(inference={k:v.tolist() for k,v in result.items()},metrics=metrics,baseline=scored['always_intact']['metrics'],calibration=bins,public_status_lags=r['public_state_lags'],dictionary=d,all_filter_prefixes=120,scientific_acceptance=False,new_synthetic_data=False)
(session.output/'consumer-summary.json').write_text(json.dumps(checked,indent=2,allow_nan=False)+'\n')
summary=session.finish(dict(saved_filter_parity=True,all_prefixes_causal=True,truth_excluded_from_inference=True,smoother_ex_post=True,legacy_scoring_preserved=True,dictionary_parity=True,new_synthetic_data=False,scientific_acceptance=False))
print(summary)